# Quantum Benchmarks on Hard Scheduling Instances

This notebook is the starting point for comparing the validated MILP baseline against quantum and quantum-inspired approaches on the curated hard-instance pack.

The workflow is:

1. Load reusable hard instances from `manifest.csv`.
2. Solve selected instances with Gurobi as the source-of-truth baseline.
3. Build a reduced QUBO using the same assignment variables `x[job, cluster, start]`.
4. Solve the QUBO with classical simulated annealing as a sanity check.
5. Prepare QAOA execution paths for local simulation and IBM Runtime hardware.
6. Formulate a first Pauli Correlation Encoding experiment scaffold.
7. Prepare D-Wave execution with simulated annealing fallback.

Important scope note: the first QUBO is a reduced approximation. It encodes assignment exactly, omits incompatible variables, and uses quadratic soft penalties for capacity pressure. Feasibility is checked after decoding. The MILP remains the authoritative feasibility and cost model.


In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path
import sys
import time
from typing import Any

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import ModelConfig
from src.evaluation.metrics import compute_summary_metrics
from src.evaluation.results import extract_cluster_hourly_results, extract_hourly_results, extract_schedule
from src.instance_generator.hard_instances import create_hard_benchmark_pack
from src.milp.gurobi_model import build_milp_model
from src.milp.solve import solve_model
from src.quantum.qubo_builder import (
    QuboPenaltyWeights,
    SchedulingQubo,
    build_scheduling_qubo,
    decode_qubo_sample,
    qubo_energy,
    validate_decoded_schedule,
)


## Configuration

`RUN_*` flags are intentionally conservative. Start with Gurobi and simulated annealing. Enable IBM or D-Wave only after setting credentials and selecting a tiny quantum slice.


In [2]:
BENCHMARK_DIR = PROJECT_ROOT / "experiments" / "combinatorial_difficulty" / "constrained_benchmark_instances"
MANIFEST_PATH = BENCHMARK_DIR / "manifest.csv"
OUTPUT_DIR = PROJECT_ROOT / "experiments" / "quantum_hard_benchmarks"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Full hard instances are good for MILP/GA. Quantum hardware needs a much smaller slice.
SELECTED_INSTANCE_COUNT = 1
QUANTUM_MAX_JOBS = 5
QUANTUM_MAX_QUBO_VARIABLES = 18

GUROBI_TIME_LIMIT_SECONDS = 60
GUROBI_MIP_GAP = 0.001

RUN_GUROBI_BASELINE = True
RUN_SIMULATED_ANNEALING = True
RUN_LOCAL_QAOA_SIMULATOR = False
RUN_IBM_QAOA = False
RUN_DWAVE_HARDWARE = False

IBM_BACKEND_NAME = None  # example: "ibm_brisbane"
DWAVE_SOLVER = None      # example: "Advantage_system6.4"

PENALTY_WEIGHTS = QuboPenaltyWeights(
    assignment=200.0,
    power_capacity=80.0,
    gpu_capacity=80.0,
    cpu_capacity=30.0,
    memory_capacity=30.0,
    peak_smoothing=1.0,
)


## Load Hard Instances

If the benchmark pack does not exist yet, this cell creates it using the shared generator module. The manifest is the authoritative list of instances; do not glob every folder because older experimental folders may exist in the output directory.


In [3]:
if not MANIFEST_PATH.exists():
    manifest_df = create_hard_benchmark_pack(BENCHMARK_DIR)
else:
    manifest_df = pd.read_csv(MANIFEST_PATH)

manifest_df = manifest_df.sort_values(
    ["global_gpu_slot_utilization", "cheap_block_pressure_actual", "assignment_density_proxy"],
    ascending=[False, False, False],
).reset_index(drop=True)

selected_manifest_df = manifest_df.head(SELECTED_INSTANCE_COUNT).copy()
selected_manifest_df[[
    "instance_name",
    "num_jobs",
    "global_gpu_slot_utilization",
    "cheap_block_pressure_actual",
    "restricted_gpu_share_actual",
    "multi_gpu_share_actual",
    "assignment_density_proxy",
]]


,instance_name,num_jobs,global_gpu_slot_utilization,cheap_block_pressure_actual,restricted_gpu_share_actual,multi_gpu_share_actual,assignment_density_proxy
0,006_high_util_restricted_multi03_pressure17_se...,76,0.901042,2.883333,0.421053,0.157895,763.0


In [4]:
def load_instance(instance_dir: Path) -> dict[str, Any]:
    """Load one exported benchmark instance from disk."""

    metadata = json.loads((instance_dir / "metadata.json").read_text())
    return {
        "instance_dir": instance_dir,
        "metadata": metadata,
        "jobs": pd.read_csv(instance_dir / "jobs.csv"),
        "hourly": pd.read_csv(instance_dir / "hourly.csv"),
        "clusters": pd.read_csv(instance_dir / "clusters.csv"),
        "hidden_schedule": pd.read_csv(instance_dir / "hidden_feasible_schedule.csv"),
        "model_config": ModelConfig(**metadata["model_config"]),
    }

instances = [load_instance(Path(row.instance_dir)) for row in selected_manifest_df.itertuples(index=False)]
instances[0]["jobs"].head()


,job_id,category,workload_family,duration,power,earliest_start,latest_start,gpu_type_required,gpu_count_required,cpu_required,memory_required_gb
0,job_000,gpu_unrestricted,combinatorial_difficulty,1,0.000076,1,9,NaN,1,7.278281,49.607148
1,job_001,gpu_restricted,combinatorial_difficulty,2,0.000333,3,11,G2,1,9.453926,54.194537
2,job_002,gpu_unrestricted,combinatorial_difficulty,3,0.000236,4,13,NaN,1,7.451545,39.135860
3,job_003,gpu_restricted,combinatorial_difficulty,1,0.000226,3,11,T4,3,19.473792,140.280420
4,job_004,gpu_unrestricted,combinatorial_difficulty,3,0.000769,4,13,NaN,2,15.396294,80.001862


## Gurobi Baseline

This gives the exact MILP baseline for the same exported instance files. These results become the reference objective, schedule, resource utilization, and MIP gap.


In [5]:
def solve_instance_with_gurobi(instance: dict[str, Any]) -> dict[str, Any]:
    """Solve one hard instance with the canonical MILP model."""

    model, variables = build_milp_model(
        instance["jobs"],
        instance["hourly"],
        instance["clusters"],
        instance["model_config"],
        model_name=f"milp_{instance['metadata']['instance_name']}",
        enforce_gpu_constraints=True,
        enforce_cpu_constraints=True,
        enforce_memory_constraints=True,
    )
    model.Params.OutputFlag = 0
    start = time.time()
    solve_error = None
    try:
        solve_model(model, time_limit=GUROBI_TIME_LIMIT_SECONDS, mip_gap=GUROBI_MIP_GAP)
    except RuntimeError as exc:
        # Hard instances may hit the time limit before finding an incumbent.
        # Keep the run as a recorded baseline outcome and continue to QUBO sections.
        solve_error = str(exc)
    wall_s = time.time() - start

    has_solution = model.SolCount > 0
    row = {
        "instance_name": instance["metadata"]["instance_name"],
        "status": int(model.Status),
        "has_solution": bool(has_solution),
        "runtime_s": float(model.Runtime),
        "wall_s": wall_s,
        "num_vars": int(model.NumVars),
        "num_constraints": int(model.NumConstrs),
        "assignment_vars": len(variables["x"]),
        "solve_error": solve_error,
    }
    if has_solution:
        hourly_results = extract_hourly_results(instance["hourly"], variables)
        cluster_results = extract_cluster_hourly_results(variables)
        schedule = extract_schedule(instance["jobs"], variables)
        row.update({
            "objective": float(model.ObjVal),
            "best_bound": float(model.ObjBound),
            "mip_gap": float(model.MIPGap),
            **compute_summary_metrics(hourly_results, instance["model_config"]),
        })
        run_dir = OUTPUT_DIR / instance["metadata"]["instance_name"] / "gurobi"
        run_dir.mkdir(parents=True, exist_ok=True)
        schedule.to_csv(run_dir / "schedule.csv", index=False)
        hourly_results.to_csv(run_dir / "hourly_results.csv", index=False)
        cluster_results.to_csv(run_dir / "cluster_hourly_results.csv", index=False)
    return row

if RUN_GUROBI_BASELINE:
    gurobi_rows = [solve_instance_with_gurobi(instance) for instance in instances]
    gurobi_results_df = pd.DataFrame(gurobi_rows)
    gurobi_results_df.to_csv(OUTPUT_DIR / "gurobi_baseline_results.csv", index=False)
else:
    gurobi_results_df = pd.DataFrame()

gurobi_results_df


Set parameter Username


Set parameter LicenseID to value 2828833


Academic license - for non-commercial use only - expires 2027-05-29


,instance_name,status,has_solution,runtime_s,wall_s,num_vars,num_constraints,assignment_vars,solve_error
0,006_high_util_restricted_multi03_pressure17_se...,9,False,60.128192,60.128373,1767,381,1733,Gurobi finished with status 9 and no feasible ...


## Quantum Slice

The full hard instances are intentionally too large for near-term QAOA. This helper selects a small subset of jobs from one hard instance while preserving the same cluster table, energy landscape, and job structure. The slice is only for quantum feasibility experiments, not for final business-scale benchmarking.


In [6]:
def make_quantum_slice(instance: dict[str, Any], max_jobs: int, max_qubo_variables: int) -> dict[str, Any]:
    """Select a small job subset whose QUBO fits the configured variable budget."""

    jobs = instance["jobs"].copy()
    # Prefer jobs with restricted GPU requirements and multi-GPU demand because they are more combinatorial.
    jobs["_restricted"] = jobs["gpu_type_required"].fillna("").astype(str).str.len() > 0
    jobs["_multi"] = jobs["gpu_count_required"] > 1
    jobs = jobs.sort_values(["_multi", "_restricted", "duration"], ascending=[False, False, False])

    selected_jobs = []
    for _, row in jobs.iterrows():
        candidate = pd.DataFrame(selected_jobs + [row.drop(labels=["_restricted", "_multi"]).to_dict()])
        candidate_qubo = build_scheduling_qubo(
            candidate,
            instance["hourly"],
            instance["clusters"],
            instance["model_config"],
            weights=PENALTY_WEIGHTS,
        )
        if len(candidate) <= max_jobs and candidate_qubo.num_variables <= max_qubo_variables:
            selected_jobs.append(row.drop(labels=["_restricted", "_multi"]).to_dict())
        if len(selected_jobs) >= max_jobs:
            break

    sliced_jobs = pd.DataFrame(selected_jobs).sort_values("job_id").reset_index(drop=True)
    qubo = build_scheduling_qubo(
        sliced_jobs,
        instance["hourly"],
        instance["clusters"],
        instance["model_config"],
        weights=PENALTY_WEIGHTS,
    )
    return {**instance, "jobs": sliced_jobs, "qubo": qubo}

quantum_instance = make_quantum_slice(instances[0], QUANTUM_MAX_JOBS, QUANTUM_MAX_QUBO_VARIABLES)
qubo = quantum_instance["qubo"]
{
    "instance_name": quantum_instance["metadata"]["instance_name"],
    "quantum_jobs": len(quantum_instance["jobs"]),
    "qubo_variables": qubo.num_variables,
    "linear_terms": len(qubo.linear),
    "quadratic_terms": len(qubo.quadratic),
    "offset": qubo.offset,
}


{'instance_name': '006_high_util_restricted_multi03_pressure17_seed_93001',
 'quantum_jobs': 2,
 'qubo_variables': 17,
 'linear_terms': 17,
 'quadratic_terms': 88,
 'offset': 400.0}

## QUBO Formulation

The QUBO has binary variables:

\[
z_a \equiv x_{i,k,s}\in\{0,1\}
\]

where variable `a` represents a feasible tuple `(job i, cluster k, start s)`.

The current reduced Hamiltonian is:

\[
H(z)=H_{energy}(z)+H_{assign}(z)+H_{capacity-proxy}(z)+H_{smooth}(z)
\]

The exact assignment penalty is:

\[
H_{assign}=\lambda_A\sum_i\left(1-\sum_{k,s}x_{i,k,s}\right)^2
\]

Compatibility is handled by variable omission: incompatible `(job, cluster, start)` variables are not created.

Capacity terms are soft quadratic proxies:

\[
H_{capacity-proxy}=\sum_{k,t,r}\lambda_r\left(L_{k,t,r}(z)-C_{k,r}\right)^2
\]

This is not yet an exact inequality encoding. Therefore every decoded solution is validated against the exact resource constraints after sampling. Infeasible decoded samples are rejected or repaired in later work.


In [7]:
qubo_variables_df = pd.DataFrame(qubo.variables)
qubo_variables_df.head(20)


,job_id,cluster,start,duration,power,gpu_count_required,cpu_required,memory_required_gb
0,job_007,cluster_t4,1,2,0.000135,2.0,17.625091,85.015155
1,job_007,cluster_t4,2,2,0.000135,2.0,17.625091,85.015155
2,job_007,cluster_t4,3,2,0.000135,2.0,17.625091,85.015155
3,job_007,cluster_t4,4,2,0.000135,2.0,17.625091,85.015155
4,job_007,cluster_t4,5,2,0.000135,2.0,17.625091,85.015155
5,job_007,cluster_t4,6,2,0.000135,2.0,17.625091,85.015155
6,job_007,cluster_t4,7,2,0.000135,2.0,17.625091,85.015155
7,job_007,cluster_t4,8,2,0.000135,2.0,17.625091,85.015155
8,job_007,cluster_t4,9,2,0.000135,2.0,17.625091,85.015155
9,job_007,cluster_t4,10,2,0.000135,2.0,17.625091,85.015155


## Classical QUBO Sanity Check

Before using QAOA or D-Wave, solve the QUBO with classical simulated annealing. This validates coefficient construction, decoding, and feasibility checks.


In [8]:
def to_dimod_bqm(qubo: SchedulingQubo):
    """Convert SchedulingQubo into a dimod BinaryQuadraticModel."""

    import dimod

    bqm = dimod.BinaryQuadraticModel({}, {}, qubo.offset, dimod.BINARY)
    for index, coefficient in qubo.linear.items():
        bqm.add_variable(index, coefficient)
    for (left, right), coefficient in qubo.quadratic.items():
        bqm.add_interaction(left, right, coefficient)
    return bqm


def solve_qubo_with_neal(qubo: SchedulingQubo, reads: int = 500) -> dict[str, Any]:
    """Solve the QUBO with simulated annealing from D-Wave Ocean's neal package."""

    import neal

    bqm = to_dimod_bqm(qubo)
    sampler = neal.SimulatedAnnealingSampler()
    sampleset = sampler.sample(bqm, num_reads=reads)
    best = sampleset.first
    sample = {int(variable): int(value) for variable, value in best.sample.items()}
    schedule = decode_qubo_sample(qubo, sample)
    report = validate_decoded_schedule(
        schedule,
        quantum_instance["jobs"],
        quantum_instance["clusters"],
        quantum_instance["hourly"]["hour"].astype(int).tolist(),
    )
    return {
        "energy": float(best.energy),
        "num_occurrences": int(best.num_occurrences),
        "sample": sample,
        "schedule": schedule,
        "feasibility_report": report,
    }

if RUN_SIMULATED_ANNEALING:
    annealing_result = solve_qubo_with_neal(qubo, reads=1000)
else:
    annealing_result = None

annealing_result["feasibility_report"] if annealing_result else "Simulated annealing disabled"


{'feasible': True, 'violations': []}

In [9]:
if annealing_result:
    display(annealing_result["schedule"])
    print("QUBO energy:", annealing_result["energy"])


,job_id,assigned_cluster,start_hour,duration,power,gpu_count_required,cpu_required,memory_required_gb,qubo_variable
0,job_007,cluster_t4,6,2,0.000135,2.0,17.625091,85.015155,5
1,job_012,cluster_t4,9,3,0.000217,3.0,23.683531,123.066082,14


QUBO energy: 0.009648689870118687


## QAOA Formulation

QAOA requires converting the QUBO into an Ising Hamiltonian. For binary variable `z_i`, use:

\[
z_i = \frac{1-Z_i}{2}
\]

The resulting operator is a weighted sum of identity, single-qubit `Z_i`, and two-qubit `Z_iZ_j` Pauli terms. The number of QAOA qubits equals the number of QUBO binary variables, so use only small slices.


In [10]:
def qubo_to_sparse_pauli_op(qubo: SchedulingQubo):
    """Convert QUBO coefficients into a Qiskit SparsePauliOp Ising Hamiltonian."""

    from qiskit.quantum_info import SparsePauliOp

    n = qubo.num_variables
    terms: dict[str, float] = {"I" * n: qubo.offset}

    def add_pauli(label: str, coeff: float) -> None:
        terms[label] = terms.get(label, 0.0) + float(coeff)

    for i, coeff in qubo.linear.items():
        # q_i z_i = q_i/2 - q_i/2 Z_i
        add_pauli("I" * n, coeff / 2.0)
        label = ["I"] * n
        label[n - 1 - i] = "Z"
        add_pauli("".join(label), -coeff / 2.0)

    for (i, j), coeff in qubo.quadratic.items():
        # q_ij z_i z_j = q_ij/4 (I - Z_i - Z_j + Z_i Z_j)
        add_pauli("I" * n, coeff / 4.0)
        label_i = ["I"] * n
        label_i[n - 1 - i] = "Z"
        add_pauli("".join(label_i), -coeff / 4.0)
        label_j = ["I"] * n
        label_j[n - 1 - j] = "Z"
        add_pauli("".join(label_j), -coeff / 4.0)
        label_ij = ["I"] * n
        label_ij[n - 1 - i] = "Z"
        label_ij[n - 1 - j] = "Z"
        add_pauli("".join(label_ij), coeff / 4.0)

    labels = []
    coeffs = []
    for label, coeff in terms.items():
        if abs(coeff) > 1e-12:
            labels.append(label)
            coeffs.append(coeff)
    return SparsePauliOp(labels, coeffs)

ising_operator = qubo_to_sparse_pauli_op(qubo)
{
    "qubits": qubo.num_variables,
    "pauli_terms": len(ising_operator),
}


{'qubits': 17, 'pauli_terms': 106}

In [11]:
def run_local_qaoa_placeholder(qubo: SchedulingQubo) -> dict[str, Any]:
    """Run a local QAOA simulation when qiskit-algorithms setup is available."""

    # This is intentionally isolated because Qiskit algorithm APIs change across versions.
    # If this cell fails, keep the Ising operator above as the stable formulation artifact.
    from qiskit_algorithms import QAOA
    from qiskit_algorithms.optimizers import COBYLA
    from qiskit.primitives import StatevectorSampler

    operator = qubo_to_sparse_pauli_op(qubo)
    sampler = StatevectorSampler()
    qaoa = QAOA(sampler=sampler, optimizer=COBYLA(maxiter=50), reps=1)
    result = qaoa.compute_minimum_eigenvalue(operator)
    return {"eigenvalue": result.eigenvalue, "raw_result": result}

if RUN_LOCAL_QAOA_SIMULATOR:
    if qubo.num_variables > 18:
        raise ValueError("Reduce QUANTUM_MAX_QUBO_VARIABLES before running local QAOA.")
    local_qaoa_result = run_local_qaoa_placeholder(qubo)
else:
    local_qaoa_result = None

local_qaoa_result


## IBM Runtime QAOA Placeholder

Set `RUN_IBM_QAOA = True`, export your IBM token as `IBM_QUANTUM_TOKEN`, and optionally set `IBM_BACKEND_NAME`. Keep the QUBO slice very small. Hardware execution should be treated as a feasibility demonstration, not a performance comparison, until embedding, transpilation, shot count, and queue effects are controlled.


In [12]:
def run_ibm_qaoa_placeholder(qubo: SchedulingQubo, backend_name: str | None = None) -> dict[str, Any]:
    """Prepare an IBM Runtime execution context for QAOA experiments."""

    token = os.environ.get("IBM_QUANTUM_TOKEN")
    if not token:
        raise RuntimeError("Set IBM_QUANTUM_TOKEN before enabling IBM hardware execution.")

    from qiskit_ibm_runtime import QiskitRuntimeService

    service = QiskitRuntimeService(channel="ibm_quantum", token=token)
    backend = service.backend(backend_name) if backend_name else service.least_busy(operational=True, simulator=False)
    operator = qubo_to_sparse_pauli_op(qubo)
    return {
        "backend_name": backend.name,
        "num_qubits_required": qubo.num_variables,
        "num_pauli_terms": len(operator),
        "next_step": "Create/pass manager + sampler/estimator workflow for the selected backend.",
    }

if RUN_IBM_QAOA:
    ibm_qaoa_plan = run_ibm_qaoa_placeholder(qubo, IBM_BACKEND_NAME)
else:
    ibm_qaoa_plan = {"status": "disabled", "credential_env_var": "IBM_QUANTUM_TOKEN"}

ibm_qaoa_plan


{'status': 'disabled', 'credential_env_var': 'IBM_QUANTUM_TOKEN'}

## Pauli Correlation Encoding Scaffold

PCE does not map one binary variable to one qubit directly. Instead, each binary decision is decoded from the sign or squashed expectation of a Pauli-string observable:

\[
z_i \approx \frac{1-\tanh(\alpha \langle P_i \rangle)}{2}
\]

The practical experiment plan is:

1. Assign each QUBO variable to a Pauli string over fewer qubits.
2. Optimize a parameterized quantum state against the QUBO energy evaluated on decoded relaxed variables.
3. Increase `alpha` iteratively to force more binary-like decoded values.
4. Decode bitstrings, validate feasibility, and compare against the MILP baseline.

The cell below creates deterministic Pauli-string assignments and evaluates the relaxed QUBO objective for a vector of Pauli expectations. This is the formulation anchor; the variational circuit optimizer can be added after the reduced QUBO is stable.


In [13]:
def make_pce_pauli_map(num_variables: int, num_qubits: int, seed: int = 7) -> list[str]:
    """Assign each QUBO variable to a deterministic random Pauli string."""

    rng = np.random.default_rng(seed)
    paulis = np.array(["I", "X", "Y", "Z"])
    mapping = []
    while len(mapping) < num_variables:
        label = "".join(rng.choice(paulis, size=num_qubits, p=[0.25, 0.25, 0.25, 0.25]))
        if set(label) != {"I"} and label not in mapping:
            mapping.append(label)
    return mapping


def relaxed_qubo_energy(qubo: SchedulingQubo, relaxed_bits: np.ndarray) -> float:
    """Evaluate QUBO on relaxed variables in [0, 1]."""

    energy = qubo.offset
    energy += sum(coeff * relaxed_bits[index] for index, coeff in qubo.linear.items())
    energy += sum(coeff * relaxed_bits[left] * relaxed_bits[right] for (left, right), coeff in qubo.quadratic.items())
    return float(energy)


def pce_decode_expectations(expectations: np.ndarray, alpha: float) -> np.ndarray:
    """Decode Pauli expectations into relaxed binary variables."""

    return (1.0 - np.tanh(alpha * expectations)) / 2.0

PCE_NUM_QUBITS = min(6, max(2, int(np.ceil(np.log2(max(qubo.num_variables, 2))))))
pce_pauli_map = make_pce_pauli_map(qubo.num_variables, PCE_NUM_QUBITS)
example_expectations = np.linspace(-0.8, 0.8, qubo.num_variables)
pce_relaxed_bits = pce_decode_expectations(example_expectations, alpha=2.0)
{
    "qubo_variables": qubo.num_variables,
    "pce_qubits": PCE_NUM_QUBITS,
    "first_pauli_strings": pce_pauli_map[:5],
    "example_relaxed_energy": relaxed_qubo_energy(qubo, pce_relaxed_bits),
}


{'qubo_variables': 17,
 'pce_qubits': 5,
 'first_pauli_strings': ['YZZIX', 'ZIZZX', 'XXXXY', 'YZZYZ', 'IIYII'],
 'example_relaxed_energy': 9612.223517577975}

## D-Wave Annealing

The fallback path uses `neal` simulated annealing. The hardware path is left credential-gated because solver access may not be available yet. Once access is granted, set `DWAVE_API_TOKEN` and `RUN_DWAVE_HARDWARE = True`.


In [14]:
def run_dwave_hardware_placeholder(qubo: SchedulingQubo, solver: str | None = None) -> dict[str, Any]:
    """Submit the QUBO to a D-Wave sampler when credentials are available."""

    token = os.environ.get("DWAVE_API_TOKEN")
    if not token:
        raise RuntimeError("Set DWAVE_API_TOKEN before enabling D-Wave hardware execution.")

    from dwave.system import DWaveSampler, EmbeddingComposite

    bqm = to_dimod_bqm(qubo)
    sampler_kwargs = {"token": token}
    if solver:
        sampler_kwargs["solver"] = solver
    sampler = EmbeddingComposite(DWaveSampler(**sampler_kwargs))
    sampleset = sampler.sample(bqm, num_reads=100)
    best = sampleset.first
    return {
        "energy": float(best.energy),
        "sample": {int(k): int(v) for k, v in best.sample.items()},
        "info": sampleset.info,
    }

if RUN_DWAVE_HARDWARE:
    dwave_result = run_dwave_hardware_placeholder(qubo, DWAVE_SOLVER)
else:
    dwave_result = {"status": "disabled", "fallback": "neal simulated annealing already available"}

dwave_result


{'status': 'disabled',
 'fallback': 'neal simulated annealing already available'}

## Save Quantum Benchmark Artifacts

This writes the reduced quantum slice, QUBO coefficients, variables, and any simulated annealing schedule so future notebooks can reuse the same problem instance.


In [15]:
artifact_dir = OUTPUT_DIR / quantum_instance["metadata"]["instance_name"] / "quantum_slice"
artifact_dir.mkdir(parents=True, exist_ok=True)

quantum_instance["jobs"].to_csv(artifact_dir / "jobs.csv", index=False)
quantum_instance["hourly"].to_csv(artifact_dir / "hourly.csv", index=False)
quantum_instance["clusters"].to_csv(artifact_dir / "clusters.csv", index=False)
pd.DataFrame(qubo.variables).to_csv(artifact_dir / "qubo_variables.csv", index=False)
pd.DataFrame([{"index": k, "coefficient": v} for k, v in qubo.linear.items()]).to_csv(artifact_dir / "qubo_linear.csv", index=False)
pd.DataFrame([{"left": k[0], "right": k[1], "coefficient": v} for k, v in qubo.quadratic.items()]).to_csv(artifact_dir / "qubo_quadratic.csv", index=False)

metadata = {
    "source_instance": quantum_instance["metadata"]["instance_name"],
    "quantum_jobs": len(quantum_instance["jobs"]),
    "qubo_variables": qubo.num_variables,
    "qubo_offset": qubo.offset,
    "penalty_weights": PENALTY_WEIGHTS.__dict__,
    "qaoa_qubits_required": qubo.num_variables,
    "pce_qubits_demo": PCE_NUM_QUBITS,
}
(artifact_dir / "metadata.json").write_text(json.dumps(metadata, indent=2))

if annealing_result:
    annealing_result["schedule"].to_csv(artifact_dir / "neal_schedule.csv", index=False)
    (artifact_dir / "neal_result.json").write_text(json.dumps({
        "energy": annealing_result["energy"],
        "num_occurrences": annealing_result["num_occurrences"],
        "feasibility_report": annealing_result["feasibility_report"],
    }, indent=2))

artifact_dir


PosixPath('/Users/juanparrado/Documents/Quantum Master/master_thesis/data_center_scheduling/experiments/quantum_hard_benchmarks/006_high_util_restricted_multi03_pressure17_seed_93001/quantum_slice')